# PG-MoE — Phase Arbitrator 构建 + 可视化

这个 notebook 构建 **Phase Arbitrator**——PG-MoE 整个项目最关键的 novelty 模块。

**和 `train_imu.ipynb` 的本质区别：**

| | IMU Expert | Phase Arbitrator |
|---|---|---|
| 有监督信号？ | 有（27 类标签） | **没有** |
| 怎么训练？ | 直接 CE loss | 只能 joint with PG-MoE |
| 输入 | 原始 IMU | **原始 IMU**（不是 expert 输出） |
| 输出 | tokens `(B, 12, 256)` | α(t) `(B, 12)` |

所以这个 notebook **不训练任何东西**，它做三件事：
1. 构建模块（定义网络架构）
2. 可视化物理特征（验证我们抽的东西是对的）
3. 可视化未训练的 α(t)（确认 shape 和数值范围合理）

真正的训练发生在后面 `pgmoe.py` 的联合训练里。

---

## 1. 挂载 Drive + 配置

和前面 notebook 一样。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

DATA_ROOT = "/content/drive/MyDrive/utd_mhad"
SAVE_DIR  = "/content/drive/MyDrive/pgmoe_ckpt"

IMU_LEN = 192
T_I = 12               # IMU token 数（和 IMU expert 一致）

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 2. 加载一个具体动作样本

我们挑 **"right arm throw"（action 5 in filename, class 4）**——这是 IMU 单模态最差的类（acc=25%）。

**为什么挑最差的？** 因为 PG-MoE 的卖点是"在 IMU 失灵的时刻让 vision 接管"。如果我们能把 throw 这个样本的物理特征画出来，看到清晰的 "prep / accel / impact / follow" 几个阶段，就能说服自己 phase arbitrator 应该让 α 在 impact 那一瞬间下降（让 vision 接管）。

In [ ]:
# 找几个 throw 样本
throw_files = sorted(glob.glob(os.path.join(DATA_ROOT, "Inertial", "a5_s1_*_inertial.mat")))
print(f"Found {len(throw_files)} throw files for subject 1")
for f in throw_files:
    print("  ", os.path.basename(f))

# 选第一个
sample_path = throw_files[0]
raw = sio.loadmat(sample_path)["d_iner"].astype(np.float32)
print(f"\nSelected: {os.path.basename(sample_path)}")
print(f"Raw shape: {raw.shape}  (timesteps, 6 channels)")

# Pad / truncate 到 192
if raw.shape[0] < IMU_LEN:
    pad = np.zeros((IMU_LEN - raw.shape[0], 6), np.float32)
    raw = np.concatenate([raw, pad], axis=0)
raw = raw[:IMU_LEN]
print(f"After pad/crop: {raw.shape}")

# Transpose 成 (6, 192) 给 PyTorch
imu = torch.from_numpy(raw).T.contiguous().unsqueeze(0)  # (1, 6, 192)
print(f"PyTorch tensor: {tuple(imu.shape)}")

## 3. 画原始 6 通道

前 3 个通道是 acc_x/y/z（加速度计），后 3 个是 gyro_x/y/z（陀螺仪）。

**看什么：** 找到 throw 动作的 "准备 → 加速 → 冲击 → 跟随" 这几个阶段，看看哪些通道在哪些时刻有大动静。

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
ch_names = ["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]
for i in range(3):
    axes[0].plot(raw[:, i], label=ch_names[i])
axes[0].set_title("Accelerometer channels (raw IMU, throw action)")
axes[0].legend(loc="upper right"); axes[0].grid(True, alpha=0.3)

for i in range(3, 6):
    axes[1].plot(raw[:, i], label=ch_names[i])
axes[1].set_title("Gyroscope channels")
axes[1].set_xlabel("timestep (1.9 s @ 100 Hz)")
axes[1].legend(loc="upper right"); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 4. 三个物理特征——为什么是这三个？

Phase Arbitrator 不直接吃原始 6 通道，而是先从中**抽出三个高层物理量**，然后让 MLP 看这三个量决定 α(t)。

| 特征 | 公式 | 物理意义 | 什么时候大 |
|---|---|---|---|
| **加速度模** | √(a_x²+a_y²+a_z²) | 整体运动剧烈程度（与方向无关） | 加速 / 冲击阶段 |
| **二阶导数** | d²\|a\|/dt² | 加速度变化的快慢（jerk-like） | phase 转换点 |
| **能量变化率** | d/dt(\|a\|²) | 动能演变 | 能量注入/释放 |

**关键设计选择：为什么不让 MLP 吃 deep feature？**

如果输入是 IMU expert 学出来的 256 维 deep feature，那 α 就是"从模型学到的高层特征里再学一个 gating"——这就退化成 MMTSA / DynMM 那种**黑盒 attention**了。

用**物理量**做输入有三个好处：
1. **可解释**：α 由具体的物理信号驱动，可以画出来给人看
2. **可验证**：能对照 midterm crossover curve 检查 α 是否符合物理直觉
3. **不依赖 IMU expert 的训练好坏**：即使 IMU expert 还没收敛，物理特征也是稳定的

## 5. 计算并画三个特征

**实现细节：**
- 用 `torch.diff` 做差分（一阶导）
- 对差分结果再做 `torch.diff` 拿到二阶导
- `prepend=...[:1]` 是让差分保持原长度（否则每次差分会少一个点）

In [ ]:
acc = imu[:, :3]                                          # (1, 3, 192)
mag = torch.sqrt((acc**2).sum(dim=1, keepdim=True) + 1e-8) # (1, 1, 192)  ← |a|

mag_d  = torch.diff(mag, dim=2, prepend=mag[:, :, :1])    # (1, 1, 192) 一阶导
mag_dd = torch.diff(mag_d, dim=2, prepend=mag_d[:, :, :1])# (1, 1, 192) 二阶导

energy      = (acc**2).sum(dim=1, keepdim=True)            # (1, 1, 192)  ← |a|²
energy_rate = torch.diff(energy, dim=2, prepend=energy[:, :, :1])  # (1,1,192) 能量变化率

# 画
fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
t = np.arange(IMU_LEN)
axes[0].plot(t, mag[0, 0].numpy());          axes[0].set_title("|a(t)| — acceleration magnitude")
axes[1].plot(t, mag_dd[0, 0].numpy(), color="C1"); axes[1].set_title("d²|a|/dt² — 2nd derivative (phase transitions)")
axes[2].plot(t, energy_rate[0, 0].numpy(), color="C2"); axes[2].set_title("d/dt(|a|²) — energy rate of change")
axes[2].set_xlabel("timestep")
for ax in axes: ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 6. 把特征提取封装成 `PhaseFeatures` 模块

上面是手工算，下面写成 `nn.Module` 让它能整合进网络（gradient 能自动反传）。

注意它**没有任何可学习参数**——纯粹是物理量的计算公式。

In [ ]:
class PhaseFeatures(nn.Module):
    """Extract (mag, d2_mag, energy_rate) from raw IMU (acc channels).
    Input:  (B, 6, T)   — first 3 channels = acc_x/y/z, last 3 = gyro (ignored)
    Output: (B, 3, T)   — (|a|, d²|a|/dt², d/dt|a|²)
    No learnable parameters.
    """
    def forward(self, x):
        acc = x[:, :3]
        mag = torch.sqrt((acc**2).sum(dim=1, keepdim=True) + 1e-8)
        mag_d  = torch.diff(mag, dim=2, prepend=mag[:, :, :1])
        mag_dd = torch.diff(mag_d, dim=2, prepend=mag_d[:, :, :1])
        energy = (acc**2).sum(dim=1, keepdim=True)
        e_rate = torch.diff(energy, dim=2, prepend=energy[:, :, :1])
        return torch.cat([mag, mag_dd, e_rate], dim=1)

pf = PhaseFeatures()
feats = pf(imu)
print("Phase features shape:", tuple(feats.shape))   # 期望 (1, 3, 192)

## 7. 下采样到 T_i=12

现在物理特征是 `(1, 3, 192)`——192 个时间点。但 IMU expert 输出的 token 是 12 个，α(t) 也要在 12 个时间步上输出（这样才能逐 token 加权融合）。

用 `AdaptiveAvgPool1d(12)` 把 192 个点平均池化成 12 个段，每个段约 0.16 秒的均值。

**为什么 avg pool 而不是 strided conv？** 物理量本身就是连续平滑的信号，简单平均能保留 phase 结构。Strided conv 会引入可学习的滤波器，可能不必要地复杂化。

In [ ]:
pool = nn.AdaptiveAvgPool1d(T_I)
feats_pooled = pool(feats)
print("After pool:", tuple(feats_pooled.shape))   # 期望 (1, 3, 12)

# 把高分辨率和低分辨率对照画一下
fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=False)
names = ["|a|", "d²|a|/dt²", "energy_rate"]
for i in range(3):
    axes[i].plot(np.arange(IMU_LEN), feats[0, i].numpy(), alpha=0.5, label="192 raw")
    axes[i].plot(np.linspace(0, IMU_LEN, T_I), feats_pooled[0, i].numpy(),
                 "o-", color="red", label=f"{T_I} pooled")
    axes[i].set_title(names[i]); axes[i].legend(loc="upper right"); axes[i].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 8. Phase Encoder：3 维物理特征 → 32 维 phase embedding

plan 第二节模块 4：
```
Phase Encoder:
  Linear(F→64) → ReLU → Linear(64→32) → ReLU
```

**实现技巧：** 我们用 `Conv1d(in, out, kernel_size=1)`，效果完全等同于在每个时间步独立做 `Linear`，但保持 `(B, C, T)` 格式更方便。这是 1D 网络里很标准的写法。

**为什么先升到 64 再降到 32？** 这是 MLP 的标准 bottleneck：先扩展到高维度让模型有空间组合特征，再压缩到低维度强迫它学到精简的表示。3→64→32 看起来 weird 但其实参数量很小（约 200 + 2000 = 2200 个参数）。

In [ ]:
class PhaseEncoder(nn.Module):
    """3 → 64 → 32, per timestep."""
    def __init__(self, in_dim=3, hidden=64, out_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_dim, hidden, kernel_size=1), nn.ReLU(inplace=True),
            nn.Conv1d(hidden, out_dim, kernel_size=1), nn.ReLU(inplace=True),
        )
    def forward(self, x):    # x: (B, 3, T)
        return self.net(x)   # -> (B, 32, T)

enc = PhaseEncoder()
h = enc(feats_pooled)
print("Phase embedding shape:", tuple(h.shape))   # 期望 (1, 32, 12)

## 9. Arbitrator MLP：32 维 phase embedding → α ∈ [0, 1]

plan 第二节模块 4：
```
Arbitrator MLP:
  Linear(32→16) → ReLU → Linear(16→1) → Sigmoid
```

**Sigmoid 的作用**：把任意实数挤进 (0, 1)。这样 α 永远是合法的混合权重，融合公式 `z = α·f_v + (1-α)·f_i` 不会爆炸。

**为什么不用 softmax？** Softmax 是给多类用的。我们只有两个模态（vision / IMU），二元混合，sigmoid 就够，**而且 1-α 直接就是 IMU 的权重，比 softmax over 2 更直观**。

In [ ]:
class ArbitratorMLP(nn.Module):
    """32 → 16 → 1 → sigmoid, per timestep."""
    def __init__(self, in_dim=32, hidden=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_dim, hidden, kernel_size=1), nn.ReLU(inplace=True),
            nn.Conv1d(hidden, 1, kernel_size=1),
            nn.Sigmoid(),
        )
    def forward(self, x):    # x: (B, 32, T)
        return self.net(x)   # -> (B, 1, T)

arb = ArbitratorMLP()
alpha = arb(h)
print("Alpha shape:", tuple(alpha.shape))                 # 期望 (1, 1, 12)
print("Alpha values:", alpha.squeeze().detach().numpy())   # 每个值都 ∈ (0, 1)

## 10. 组装成完整 `PhaseArbitrator`

把上面三个模块（`PhaseFeatures` + `PhaseEncoder` + `ArbitratorMLP`）串起来。这就是要放进 `models/phase_arbitrator.py` 的最终类。

In [ ]:
class PhaseArbitrator(nn.Module):
    """Input:  raw IMU (B, 6, 192)
    Output: alpha (B, T_i)  ∈ [0, 1]   — vision weight per IMU token.
    """
    def __init__(self, T_i=12, feat_dim=3, enc_hidden=64, enc_out=32, arb_hidden=16):
        super().__init__()
        self.features = PhaseFeatures()
        self.pool = nn.AdaptiveAvgPool1d(T_i)
        self.encoder = PhaseEncoder(feat_dim, enc_hidden, enc_out)
        self.arbitrator = ArbitratorMLP(enc_out, arb_hidden)

    def forward(self, imu):                      # (B, 6, 192)
        feats = self.features(imu)               # (B, 3, 192)
        feats = self.pool(feats)                 # (B, 3, T_i)
        h = self.encoder(feats)                  # (B, 32, T_i)
        alpha = self.arbitrator(h)               # (B, 1, T_i)
        return alpha.squeeze(1)                  # (B, T_i)

pa = PhaseArbitrator(T_i=T_I)
alpha = pa(imu)
print("Full PhaseArbitrator output:", tuple(alpha.shape))   # 期望 (1, 12)
print("Alpha values (untrained):", alpha.squeeze().detach().numpy())
print("\nParams:", f"{sum(p.numel() for p in pa.parameters()):,}")

## 11. 可视化未训练的 α(t)

**注意：以下 α 是用随机权重算出来的**——sigmoid 把所有值压到 0.5 附近。

**你应该看到：**
- α 都在 0.4–0.6 之间（sigmoid 在 0 附近的输出范围）
- 形状几乎是平的，没有明显的 phase pattern

**这是预期的！** 因为还没训练，模型不知道"什么时候该信谁"。等联合训练后再画一次，α 应该会出现明显的 phase-dependent 起伏。

In [ ]:
alpha_np = alpha.squeeze().detach().numpy()
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=False)
axes[0].plot(np.arange(IMU_LEN), mag[0, 0].numpy(), color="gray", label="|a(t)| (reference)")
axes[0].set_title("Raw IMU magnitude (for reference)")
axes[0].grid(True, alpha=0.3); axes[0].legend()

axes[1].plot(np.linspace(0, IMU_LEN, T_I), alpha_np, "o-", color="red")
axes[1].set_ylim(0, 1)
axes[1].axhline(0.5, ls="--", color="gray", alpha=0.5)
axes[1].set_title("α(t) — untrained (random weights, expect flat near 0.5)")
axes[1].set_xlabel("timestep (IMU resolution)")
axes[1].set_ylabel("α (1=trust vision, 0=trust IMU)")
axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 12. 多个样本的 α(t)（不同动作）

拿几个不同类别的样本——一个走路（IMU 强）、一个 throw（IMU 弱）、一个 squat（IMU 强）——把它们的 α(t) 画在一起。

**现在因为没训练，三条曲线应该都差不多**（都接近 0.5）。**等 PG-MoE 训完之后**重画这张图，正确的模型应该：
- throw 的 α 在 impact 时刻偏高（信 vision）
- walking / squat 的 α 偏低（信 IMU）

In [ ]:
def load_imu_sample(action_id, subject=1, trial=1):
    """Load a single (6, 192) IMU sample. action_id is 1-indexed (filename convention)."""
    pattern = f"a{action_id}_s{subject}_t{trial}_inertial.mat"
    fpath = os.path.join(DATA_ROOT, "Inertial", pattern)
    raw = sio.loadmat(fpath)["d_iner"].astype(np.float32)
    if raw.shape[0] < IMU_LEN:
        raw = np.concatenate([raw, np.zeros((IMU_LEN - raw.shape[0], 6), np.float32)], axis=0)
    return torch.from_numpy(raw[:IMU_LEN]).T.contiguous().unsqueeze(0)

# 选三个有代表性的动作
demo_actions = [
    (5,  "throw (IMU weak — should ↑α)"),     # class 4, worst
    (22, "walking (IMU strong — should ↓α)"),  # class 21, periodic, 100% acc
    (27, "squat (IMU strong — should ↓α)"),    # class 26, 100% acc
]

fig, ax = plt.subplots(figsize=(10, 4))
for action_id, label in demo_actions:
    try:
        imu_sample = load_imu_sample(action_id, subject=1, trial=1)
        a = pa(imu_sample).squeeze().detach().numpy()
        ax.plot(np.linspace(0, 1, T_I), a, "o-", label=label)
    except Exception as e:
        print(f"Skipped action {action_id}: {e}")

ax.axhline(0.5, ls="--", color="gray", alpha=0.5)
ax.set_ylim(0, 1)
ax.set_xlabel("normalized time within action (0=start, 1=end)")
ax.set_ylabel("α (1 = trust vision)")
ax.set_title("α(t) — untrained, all near 0.5; rerun after PG-MoE joint training")
ax.grid(True, alpha=0.3); ax.legend(loc="lower center")
plt.tight_layout(); plt.show()

## 13. 把代码同步到 repo 的 `models/phase_arbitrator.py`

上面的三个类（`PhaseFeatures` / `PhaseEncoder` / `ArbitratorMLP` / `PhaseArbitrator`）将来要被 `pgmoe.py` import。这个 cell 把它们写到 Drive 上你 repo 路径下的 `models/phase_arbitrator.py`。

**如果你的 repo 没在 Drive 上**：跳过这个 cell，等下载 notebook 后我帮你把这些类同步到本地 repo。

In [ ]:
# 改成你 Drive 上 repo 的实际路径；如果 repo 不在 Drive 上就跳过整个 cell
REPO_DRIVE_PATH = "/content/drive/MyDrive/Multi-Modal-AI"
target = os.path.join(REPO_DRIVE_PATH, "project", "final", "code", "models", "phase_arbitrator.py")

code = '''"""Phase Arbitrator for PG-MoE (Xiaoyang).

Reads raw IMU, extracts physical phase features (acc magnitude, jerk, energy rate),
and produces alpha(t) in [0, 1] that gates vision vs IMU during fusion:

    z(t) = alpha(t) * f_vision(t) + (1 - alpha(t)) * f_imu(t)

Input:  (B, 6, 192)   raw IMU
Output: (B, T_i=12)   alpha values
"""

import torch
import torch.nn as nn


class PhaseFeatures(nn.Module):
    def forward(self, x):
        acc = x[:, :3]
        mag = torch.sqrt((acc**2).sum(dim=1, keepdim=True) + 1e-8)
        mag_d  = torch.diff(mag,    dim=2, prepend=mag[:, :, :1])
        mag_dd = torch.diff(mag_d,  dim=2, prepend=mag_d[:, :, :1])
        energy = (acc**2).sum(dim=1, keepdim=True)
        e_rate = torch.diff(energy, dim=2, prepend=energy[:, :, :1])
        return torch.cat([mag, mag_dd, e_rate], dim=1)


class PhaseEncoder(nn.Module):
    def __init__(self, in_dim=3, hidden=64, out_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_dim, hidden, 1), nn.ReLU(inplace=True),
            nn.Conv1d(hidden, out_dim, 1), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)


class ArbitratorMLP(nn.Module):
    def __init__(self, in_dim=32, hidden=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_dim, hidden, 1), nn.ReLU(inplace=True),
            nn.Conv1d(hidden, 1, 1),
            nn.Sigmoid(),
        )
    def forward(self, x):
        return self.net(x)


class PhaseArbitrator(nn.Module):
    def __init__(self, T_i=12, feat_dim=3, enc_hidden=64, enc_out=32, arb_hidden=16):
        super().__init__()
        self.features = PhaseFeatures()
        self.pool = nn.AdaptiveAvgPool1d(T_i)
        self.encoder = PhaseEncoder(feat_dim, enc_hidden, enc_out)
        self.arbitrator = ArbitratorMLP(enc_out, arb_hidden)

    def forward(self, imu):
        feats = self.features(imu)
        feats = self.pool(feats)
        h = self.encoder(feats)
        alpha = self.arbitrator(h)
        return alpha.squeeze(1)
'''

if os.path.isdir(REPO_DRIVE_PATH):
    with open(target, "w") as f:
        f.write(code)
    print(f"Wrote {target}")
else:
    print(f"REPO_DRIVE_PATH not found: {REPO_DRIVE_PATH}")
    print("Skipping — the .py code is in this cell, Claude will sync it to local repo.")

## 14. 总结

**这个 notebook 做完后你拥有：**
- ✓ 一个完全定义好的 `PhaseArbitrator` 模块
- ✓ 三个物理特征的可视化图（throw 动作）
- ✓ 未训练 α(t) 的 baseline 图（应该接近 0.5）

**还没做的事：**
- ✗ 训练（必须等 PG-MoE joint training，因为没有独立标签）
- ✗ 训练后的 α(t) 可视化（等联合训练完）
- ✗ 验证 α(t) 符合 midterm 的 crossover curve（最终 ablation）

**告诉 Claude：**
1. 物理特征图看起来对不对（throw 动作有没有明显的 prep/accel/impact 峰值）
2. 未训练 α(t) 是不是都在 0.5 附近
3. 然后我们就可以进入下一阶段——**合并 PG-MoE 联合模型**（这部分必须和 Hang 协作）

---

## 还要进 git 的产物

如果你跑了 cell 13，`phase_arbitrator.py` 已经在 Drive 上更新了。否则告诉 Claude，他会把代码同步到本地 `project/final/code/models/phase_arbitrator.py`。

另外建议把 cell 5 和 cell 12 的两张可视化图保存到 `project/final/figures/`：
- `phase_features_throw.png`
- `alpha_untrained_baseline.png`

等联合训练完，重新画一张 `alpha_trained.png` 做前后对比，这是 paper 里的杀手图。